# Practical Class 2: End-to-End Classification

Welcome to the Classification Masterclass. Today, we leave the synthetic datasets behind. You will be working on a real-world, highly imbalanced business problem: **Predicting Customer Churn (or Loan Default)** https://www.kaggle.com/competitions/playground-series-s4e1.

Because we covered the mechanics of Pipelines in the last class, today's focus is heavily on **Exploratory Data Analysis (EDA)** and **Advanced Modeling**. 

---
## Part 0: Instructor Demo (30 Mins)
*Watch the screen. Your instructor will walk through the fundamentals of Classification metrics, the difference between hard labels and probabilities, and how to evaluate a model without relying on the 'Accuracy' lie.*

In [ ]:
# INSTRUCTOR USE ONLY - Live Coding Area
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, RocCurveDisplay
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# 1. Load a simple dataset
X_demo, y_demo = load_breast_cancer(return_X_y=True)
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(X_demo, y_demo, test_size=0.3, random_state=42)

# 2. Train a basic model
model = LogisticRegression(max_iter=10000).fit(X_train_d, y_train_d)

# 3. The most important distinction in classification:
hard_predictions = model.predict(X_test_d)                 # Returns [0, 1, 0, 0, 1...]
probabilities = model.predict_proba(X_test_d)[:, 1]        # Returns [0.12, 0.95, 0.33...]

# 4. Evaluation (Instructor will plot Confusion Matrix and ROC Curve here)


---
## Part 1: Exploratory Data Analysis (EDA)
It's your turn. Load the Kaggle training data (`train.csv`). 

In classification, EDA is all about seeing how features distribute *differently* across the classes. 

In [ ]:
# Load your Kaggle training data here
try:
    df = pd.read_csv('train.csv')
    print(f"Data loaded! Shape: {df.shape}")
except FileNotFoundError:
    print("Error: Ensure train.csv is in the directory.")

### Guided EDA Tasks
1. **The Target Imbalance:** Plot a count of the target variable (e.g., `Churn` or `Default`). Is the dataset balanced? If it's 90% Class 0 and 10% Class 1, what does this mean for your model?
2. **Continuous vs Target:** Choose a continuous feature (like `Age`, `Balance`, or `MonthlyCharges`). Create a `sns.boxplot()` with the target variable on the x-axis and the continuous feature on the y-axis. Do the two classes have different medians?
3. **Categorical vs Target:** Choose a categorical feature (like `ContractType` or `Education`). Create a `sns.countplot()` and use the `hue=target_column` parameter. Do certain categories have a much higher rate of the positive class?

In [ ]:
# YOUR EDA CODE HERE


### Sandbox EDA
Spend 10 minutes digging into the data. Find at least **two** features that you believe will be highly predictive of the target based on your visualizations. 

*Hint: Look for correlations, or try using `sns.kdeplot()` with `hue=target` to see overlapping distributions.*

In [ ]:
# YOUR SANDBOX EDA CODE HERE


---
## Part 2: The Preprocessing Boilerplate
Since we mastered pipelines in the last class, you don't need to rewrite the plumbing. Run the cell below to automatically build a robust preprocessing pipeline for your numeric and categorical features. 

*Make sure you define `X` (features) and `y` (target) before running this!*

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# X = ...
# y = ...

num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns

preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value='missing')), 
                      ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
])

---
## Part 3: Advanced Modeling & Imbalance

This is where competitions are won. 

**Your Task:**
1. Combine the `preprocessor` above with a classifier into a full `Pipeline`.
2. Train at least two different models (e.g., `LogisticRegression`, `RandomForestClassifier`, or `XGBClassifier`).
3. **Handle the Imbalance:** If your dataset is imbalanced, your models will struggle. Look up the `class_weight` parameter for Logistic Regression or Random Forest, or `scale_pos_weight` for XGBoost, and apply it.
4. **Evaluate correctly:** Use 5-Fold Cross-Validation, but DO NOT use Accuracy. Set the scoring parameter to `scoring='roc_auc'`. 

In [ ]:
# YOUR MODELING CODE HERE
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb 


### The Modeling Sandbox
Try to push your ROC-AUC score as high as possible. 
* Try tuning hyperparameters (`max_depth`, `n_estimators`, `C` for logistic regression).
* Check feature importances from your Random Forest to see if your EDA instincts were correct!

In [ ]:
# YOUR EXPERIMENTS HERE


---
## Part 4: The Kaggle Submission

It's time to submit to the public leaderboard. 

**CRITICAL KAGGLE CLASSIFICATION RULE:** 
Classification competitions almost always score using **ROC-AUC** or **Log-Loss**. This means Kaggle does NOT want you to submit `0` or `1`. They want you to submit the *probability* that a row is `1`. 

If you use `model.predict(test_data)`, you will submit hard labels and your score will be terrible.
You **MUST** use `model.predict_proba(test_data)[:, 1]` to get the probabilities of the positive class.

**Your Task:**
1. Fit your absolute best pipeline on the *entire* training dataset.
2. Load `test.csv`.
3. Generate PROBABILITIES for the test set using `.predict_proba()`.
4. Create a DataFrame with two columns: `Id` and `TargetColumn` (containing your probabilities).
5. Export to `submission.csv` (`index=False`) and submit to Kaggle!

In [ ]:
# 1. Load test data

# 2. Predict probabilities (Remember [:, 1] !)

# 3. Create submission DataFrame

# 4. Save to CSV


---
# The Advanced Sandbox
Did you finish early? Are you sitting at the top of the leaderboard? 

In the real world, Machine Learning Engineers don't stop at a basic Random Forest. If you want to push your skills to an industry level, attempt one (or both) of the challenges below.

## Boss Challenge 1: The Black Box is Unacceptable (SHAP)

Your model can predict *who* will churn with 85% accuracy. The CEO looks at you and asks: **"Why? What is driving our customers away?"** 

If you say "I don't know, the Random Forest just said so," you will be fired. You need **SHAP (SHapley Additive exPlanations)**. Based on game theory, SHAP breaks down the exact impact of every single feature on your model's predictions.

**Your Task:**
1. Open a terminal or add a code cell to run `!pip install shap`.
2. Extract your trained Tree model (Random Forest or XGBoost) from your pipeline.
3. Pass your model into `shap.TreeExplainer(your_model)`.
4. Calculate the SHAP values using `explainer.shap_values(X_transformed)`.
5. Generate a `shap.summary_plot()`. 

*Question to answer:* According to SHAP, what are the top 3 biggest drivers of customer churn in this bank?

In [ ]:
# !pip install shap
# import shap

# Remember: SHAP expects raw numbers, so pass your data through your preprocessor first!
# X_processed = preprocessor.transform(X)

# YOUR SHAP CODE HERE


## Boss Challenge 2: Ditch the Dinosaur (Optuna)

`GridSearchCV` is a dinosaur. It blindly guesses every possible combination of hyperparameters, wasting massive amounts of time. 

Modern ML Engineers use **Bayesian Optimization**. Instead of guessing blindly, Bayesian optimizers learn from their past guesses to "hunt" for the optimal parameters. The industry standard tool for this is **Optuna**.

**Your Task:**
1. Run `!pip install optuna`.
2. Write an `objective(trial)` function. Inside this function, ask Optuna to suggest hyperparameters (e.g., `max_depth = trial.suggest_int('max_depth', 3, 15)`).
3. Train a model with those suggested parameters and return the Cross-Validated ROC-AUC score.
4. Create an Optuna study (`optuna.create_study(direction='maximize')`) and optimize it for 20 trials.

*Hint: Optuna can tune XGBoost parameters like `learning_rate`, `n_estimators`, and `subsample` much faster than GridSearch ever could. Use this to steal 1st place on the leaderboard!*

In [ ]:
# !pip install optuna
# import optuna

# def objective(trial):
#     # 1. Suggest parameters
#     # 2. Build model
#     # 3. Return cross_val_score (roc_auc)

# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=20)

# print(study.best_params)
